# Brain Walk · Notebook 2 : les ERP et les hypothèses H1 à H10

PSY2007D — Laboratoire 1, automne 2026

<table style="width:100%; border-collapse:separate; border-spacing:0; margin:6px 0 10px 0;"><tr><td style="background:#EEF3F9; color:#0B2545; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#5B6472;">NOTEBOOK 1</div><div style="font-weight:bold; font-size:1.05em; color:#0B2545;">Prétraitement</div><div style="font-size:0.85em; color:#5B6472;">lire · nettoyer · époques</div></td><td style="background:#0B2545; color:#FFFFFF; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#C9D6E8;">NOTEBOOK 2</div><div style="font-weight:bold; font-size:1.05em; color:#FFFFFF;">ERP</div><div style="font-size:0.85em; color:#C9D6E8;">P2 · N400 · LPC · hypothèses H1-H10</div></td><td style="background:#EEF3F9; color:#0B2545; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#5B6472;">NOTEBOOK 3</div><div style="font-weight:bold; font-size:1.05em; color:#0B2545;">Rythmes</div><div style="font-size:0.85em; color:#5B6472;">spectres · temps-fréquence</div></td><td style="background:#EEF3F9; color:#0B2545; padding:10px 14px; border:3px solid #FFFFFF; border-radius:8px; vertical-align:top; width:25%;"><div style="font-size:0.8em; color:#5B6472;">NOTEBOOK 4</div><div style="font-weight:bold; font-size:1.05em; color:#0B2545;">Apprentissage</div><div style="font-size:0.85em; color:#5B6472;">décoder · ERP et rythmes</div></td></tr></table>

Le notebook 1 a nettoyé l'EEG des 17 participants (`pretraite_00/`). Ici, on teste **votre hypothèse** sur les **ERP** du groupe
(potentiels évoqués : l'EEG moyenné sur les essais, aligné sur un événement) : le comportement, les ERP, leur carte sur la tête, leur
amplitude et leur latence, l'interaction avec la posture et le lien avec le comportement.

![composantes](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/composantes.png)

Une **composante** est une onde de l'ERP. On la mesure dans une **fenêtre** (l'intervalle de temps de la mesure) et une **région** (le
groupe d'électrodes moyennées ensemble). Les composantes des hypothèses : la **P200** et la **P200-N250** (les premières étapes de la
reconnaissance du mot), la **N400** (l'accès au sens du mot), la **P3/LPC** (une positivité tardive liée à la décision) et la **CPP**
(une positivité centro-pariétale qui monte jusqu'à la réponse, mesurée avant la tape).

| Partie | Contenu |
|---|---|
| 0 | Les données, **votre hypothèse**, les options |
| 1 | Le comportement |
| 2 | Les ERP du groupe, l'onde de différence, le test par amas |
| 3 | Les topographies, et le test par amas sur toute la tête |
| 4 | Amplitude, latence et SME : **le test de votre hypothèse** |
| 5 | L'interaction avec la posture (H4, H8, H10) |
| 6 | L'EEG et le comportement (H5, H9) |
| 7 | À retenir pour votre rapport, et des pistes pour aller plus loin |
| 8 | **Défis (points bonus)** |

<div style="color:#1F2937; background:#F9FAFB; border:1px solid #E5E7EB; border-radius:6px; padding:8px 14px;"><b>Trois types de blocs</b><div style="margin:3px 0;"><span style="background:#0B5CAD; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 1</span> &nbsp;le code est complet : exécutez la cellule et lisez les commentaires.</div><div style="margin:3px 0;"><span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> &nbsp;complétez une ou deux lignes marquées <code>&lt;---</code>.</div><div style="margin:3px 0;"><span style="background:#7C3AED; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 3</span> &nbsp;écrivez la cellule vous-même, à partir des indices et des liens vers la documentation.</div><div style="font-size:0.9em; color:#5B6472; margin-top:4px;">Chaque exercice est suivi d'une solution repliée : essayez d'abord.</div></div>

<details><summary>Sur votre ordinateur ou dans Colab ?</summary>

- **Sur votre ordinateur (recommandé)** : les données sont déjà dans `tasks/brainwalk/` si vous avez fait le notebook 1.
- **Colab** : la première cellule installe MNE et télécharge les données (2,2 Go) à **chaque** session.
</details>

In [ ]:
# ---- 0.1 Les paquets (installés s'il en manque), les imports et les données
import importlib.util
import sys

DANS_COLAB = "google.colab" in sys.modules
DEPOT = "https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/"
if DANS_COLAB:                            # Colab n'a pas les fichiers du cours : on les télécharge
    !wget -q -N {DEPOT}requirements.txt {DEPOT}outils_brainwalk.py
PAQUETS = ["mne", "scipy"]   # les paquets de ce notebook
if any(importlib.util.find_spec(p) is None for p in PAQUETS):   # il en manque : installer ceux du cours (une fois)
    %pip install -q -r requirements.txt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne
from scipy import stats
import outils_brainwalk as ob             # fonctions du projet (fichier outils_brainwalk.py, à côté du notebook)

mne.set_log_level("ERROR")
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
DOSSIER = ob.telecharger_donnees()        # déjà là si vous avez fait le notebook 1
print("MNE", mne.__version__)

## 0. Les données, votre hypothèse et les options

Pour chaque **essai** utilisable (un mot ou pseudo-mot présenté, et sa réponse), le notebook 1 a découpé deux **époques** (des segments
d'EEG autour d'un événement) : autour du **mot** (−0,2 à 1,0 s) et autour de **la tape** (−0,8 à 0,2 s), le toucher du participant sur la
tablette, c'est-à-dire sa réponse. Elles sont sans correction de la ligne de base (elle se fait en 0.5).

![mot et tape](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/mot_tape.png)

- On choisit les époques d'une condition par `posture/type` : `ep["marche"]`, `ep["assis/LF"]` (type : `HF` mot fréquent, `LF` mot
  rare, `pseudo` pseudo-mot).
- La colonne `mauvaises` liste les électrodes **non mesurées** dans chaque époque (notebook 1, parties 10 et 12) : le signal d'une région est
  la moyenne de ses seules électrodes mesurées.

In [ ]:
# ---- 0.2 Charger les époques des 17 participants et le comportement
epochs, epochs_tape = {}, {}                                 # participant -> ses époques autour du mot, et autour de la tape
for fichier in sorted((DOSSIER / 'pretraite_00').glob('sub-*_epo.fif')):
    participant = fichier.name.split('_')[0][4:]             # 'sub-P003_epo.fif' -> 'P003'
    epochs[participant] = mne.read_epochs(fichier, preload=True)
    fichier_tape = DOSSIER / 'pretraite_00' / 'tape' / f'sub-{participant}_tape_epo.fif'
    if fichier_tape.exists():
        epochs_tape[participant] = mne.read_epochs(fichier_tape, preload=True)
essais = pd.read_csv(DOSSIER / 'comportement' / 'essais.csv')
print(f'{len(epochs)} participants avec de l\'EEG ({len(epochs_tape)} avec les époques de la tape), {len(essais)} essais de comportement')
comptes = pd.DataFrame({p: ep.metadata.groupby(['posture', 'type']).size() for p, ep in epochs.items()}).T.fillna(0).astype(int)
comptes                                                      # le nombre d'époques par participant et condition

### Les hypothèses H1 à H10

Chaque groupe choisit **une** hypothèse. Chacune compare deux conditions, **A** et **B**, sur une composante ; l'**effet** = B − A.

![ce que prédit chaque hypothèse](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/hypotheses_erp.png)

| | Hypothèse (version opérationnelle) | A / B | Composante | Fenêtre | Parties |
|---|---|---|---|---|---|
| H1 | La marche modifie la P200 | assis / marche | P200 | 180-250 ms | 1 à 4 |
| H2 | Les mots rares (LF) diffèrent des fréquents (HF) dès 180-300 ms | HF / LF | P200-N250 | 180-300 ms | 1 à 4 |
| H3 | La N400 est plus négative pour LF que pour HF | HF / LF | N400 | 300-500 ms | 1 à 4 |
| H4 | L'effet de fréquence sur la N400 change en marche | HF / LF, assis puis en marche | N400 | 300-500 ms | 1 à 4, puis **5** |
| H5 | L'effet N400 d'un participant suit son effet sur le comportement | HF / LF | N400 | 300-500 ms | 1 à 4, puis **6** |
| H6 | La marche modifie la P3/LPC | assis / marche | P3/LPC | 500-800 ms | 1 à 4 |
| H7 | La P3/LPC diffère entre LF et HF | HF / LF | P3/LPC | 500-800 ms | 1 à 4 |
| H8 | L'effet de fréquence sur la P3/LPC change en marche | HF / LF, assis puis en marche | P3/LPC | 500-800 ms | 1 à 4, puis **5** |
| H9 | Avant une réponse lente, la CPP monte autrement qu'avant une rapide | rapides / lents | CPP | 500 ms avant la tape | 1 à 4, puis **6** |
| H10 | La marche modifie la CPP, et son effet de fréquence | assis / marche | CPP | 500 ms avant la tape | 1 à 4, puis **5** |

<div style="border-left:5px solid #E8710A; background:#FDF1E7; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#E8710A;">À vérifier</b><br>L'énoncé officiel des hypothèses est celui du cours. Ce tableau en donne une version
opérationnelle : la mesure, la fenêtre et la région que le notebook utilise.</div>

![feuille de route](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/feuille_de_route.png)

**Écrivez votre hypothèse et vos prédictions avant de regarder les résultats.** Choisir une fenêtre après avoir vu les ERP rend le test
trop optimiste : gardez celles du tableau, ou justifiez votre choix à l'avance.

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> 0.3 Votre hypothèse

**Une ligne à changer** : `H = 'H3'`. Le tableau de la cellule est celui du dessus, écrit en Python : pour chaque hypothèse, ce qu'on
compare, la composante, sa fenêtre (en secondes) et sa région. La cellule en tire les quatre réglages du notebook.

In [ ]:
# ---- 0.3 Votre hypothèse
H = 'H3'                       # 'H1' à 'H10' : la seule ligne à changer

P200, CP = ob.REGION_P200, ob.REGION_CP    # les deux régions : F3 à P4 (9 électrodes), et la centro-pariétale C3 à P4 (6)
HYPOTHESES = {                 # H : (ce qu'on compare, la composante, la fenêtre en s, la région) : le tableau du dessus
    'H1': ('posture', 'P200', (0.18, 0.25), P200),
    'H2': ('frequence', 'P200-N250', (0.18, 0.30), P200),
    'H3': ('frequence', 'N400', (0.30, 0.50), CP),
    'H4': ('frequence', 'N400', (0.30, 0.50), CP),         # puis la partie 5
    'H5': ('frequence', 'N400', (0.30, 0.50), CP),         # puis la partie 6
    'H6': ('posture', 'P3/LPC', (0.50, 0.80), CP),
    'H7': ('frequence', 'P3/LPC', (0.50, 0.80), CP),
    'H8': ('frequence', 'P3/LPC', (0.50, 0.80), CP),       # puis la partie 5
    'H9': ('tr', 'CPP', (-0.50, 0.0), CP),                 # autour de la tape ; puis la partie 6
    'H10': ('posture', 'CPP', (-0.50, 0.0), CP),           # autour de la tape ; puis la partie 5
}
HYPOTHESE, COMPOSANTE, FENETRE, REGION = HYPOTHESES[H]
print(f"{H} : on compare « {HYPOTHESE} » sur la {COMPOSANTE}, de {FENETRE[0] * 1000:.0f} à {FENETRE[1] * 1000:.0f} ms, "
      f"région {', '.join(REGION)}")

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> 0.4 Les options (facultatif)

Pour tester votre hypothèse telle quelle, **laissez cette cellule comme elle est**. Chaque option répond à une question qu'on peut se
poser ensuite ; changez-en une, puis relancez les cellules à partir de 0.5 :

| Option | La question | Où regarder |
|---|---|---|
| `POSTURE` | L'effet existe-t-il dans une seule posture ? | parties 1 à 4 |
| `LIGNE_DE_BASE` | La correction de la ligne de base change-t-elle l'effet ? | 2.4, puis partie 4 |
| `SANS_CLIGNEMENT` | Un clignement après la réponse explique-t-il l'effet (P3/LPC, CPP) ? | parties 2 et 4 |
| `EXIGER_MINIMUM` | Les époques où la région est mal mesurée changent-elles l'effet ? | 0.6, puis partie 4 |
| `FENETRE`, `REGION` | Une autre fenêtre, ou d'autres électrodes ? **À décider avant de voir les résultats** | partout |

In [ ]:
# ---- 0.4 Les options : laissez-les telles quelles pour tester votre hypothèse
POSTURE = 'les deux'           # pour 'frequence' et 'tr' : 'les deux', 'assis' ou 'marche'
LIGNE_DE_BASE = 'soustraction' # 'soustraction' (la correction classique) ou 'regression' (Alday, 2019)
SANS_CLIGNEMENT = False        # True : seulement les essais sans clignement entre le mot et la tape
EXIGER_MINIMUM = False         # True : sans les époques où la région a moins de 4 électrodes mesurées
PARTICIPANT = 'P010'           # le participant de l'image des essais (2.3)
# FENETRE = (0.35, 0.45)       # une autre fenêtre (en s) : enlevez le # au début de la ligne
# REGION = ['Cz', 'Pz']        # d'autres électrodes

### 0.5 Préparer les époques

La cellule suivante fait cinq choses, pour chaque participant (le code : `ob.preparer_epoques`, dans `outils_brainwalk.py`) :

1. elle prend les époques autour du **mot**, ou de la **tape** pour la CPP ;
2. elle écarte les époques avec un clignement (si `SANS_CLIGNEMENT`), puis celles où la région n'a aucune électrode mesurée (moins de 4
   si `EXIGER_MINIMUM`) ;
3. pour H9, elle coupe les essais en **rapides** et **lents**, de part et d'autre du TR médian du participant (dans sa posture et son
   type de mot) ;
4. elle retire la **ligne de base** : le signal moyen de −0,2 à 0 s avant le mot (aussi pour les époques de la tape) ;
5. elle calcule le **signal de la région** dans chaque époque : la moyenne de ses électrodes mesurées.

In [ ]:
# ---- 0.5 Préparer les époques selon vos choix
POLARITE, FENETRE_LATENCE, CENTRE = ob.COMPOSANTES_ERP[COMPOSANTE]   # le sens du pic et où le chercher (partie 4) ; le zéro des temps
ZERO = 'le mot' if CENTRE == 'mot' else 'la tape'
COND_A, COND_B, CHOIX_A, CHOIX_B = ob.conditions_erp(HYPOTHESE, POSTURE)   # A et B (l'effet = B - A), et leurs époques
COULEURS = {COND_A: '#2f6db5', COND_B: '#e0791d'}
MIN_EPOQUES = 20               # un ERP de moins de 20 essais est trop bruité : le participant est laissé de côté
EPOQUES, REGIONALES, PENTES = ob.preparer_epoques(epochs, epochs_tape, CENTRE, REGION, HYPOTHESE, LIGNE_DE_BASE, SANS_CLIGNEMENT,
                                                  minimum=4 if EXIGER_MINIMUM else 1)
NOM = COMPOSANTE if (FENETRE, REGION) == HYPOTHESES[H][2:] else 'votre fenêtre'   # le nom affiché dans les figures
PARTIES = '1 à 4' + {'H4': ', puis 5', 'H8': ', puis 5', 'H10': ', puis 5', 'H5': ', puis 6', 'H9': ', puis 6'}.get(H, '')
print(f'{H} (parties {PARTIES}) : {COND_B} comparé à {COND_A} ; {NOM}, de {FENETRE[0] * 1000:.0f} à {FENETRE[1] * 1000:.0f} ms '
      f'(0 : {ZERO}), région {", ".join(REGION)} ; ligne de base : {LIGNE_DE_BASE}')

### 0.6 La région est-elle bien mesurée ?

Une électrode sèche peut perdre le contact : elle est alors **non mesurée** dans cette époque. Le signal de la région est la moyenne des
électrodes qui restent ; s'il en reste peu, il est plus bruité. Pour chaque participant et posture : les électrodes de la région jamais
mesurées, le nombre moyen d'électrodes mesurées par époque, et la part des époques qui en ont moins de 4.

**Lire.** Un participant avec beaucoup d'époques sous 4 électrodes pèse autant que les autres dans le test : `EXIGER_MINIMUM = True`
(0.4) écarte ces époques. Comparez l'effet avec et sans (partie 4).

In [ ]:
# ---- 0.6 La qualité de la région, participant par participant
ob.qualite_region(epochs_tape if CENTRE == 'tape' else epochs, REGION)

## 1. Le comportement

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Avant l'EEG, les réponses : A et B diffèrent-elles dans le comportement ? Pour chaque participant et chaque condition :

- le **TR** (temps de réaction) moyen des réponses **valides** : correctes, de 250 ms ou plus, pas aberrantes (colonne `tr_valide`) ;
- l'**exactitude** : le % de bonnes réponses ;
- l'**IES** (score d'efficacité inverse) = TR / exactitude : un TR qui « paie » les erreurs ; fiable seulement si l'exactitude dépasse
  90 %.

Le **test t apparié** compare A et B chez les mêmes participants ; **dz** = la différence moyenne / son écart-type, la taille de l'effet
(0,2 petit, 0,5 moyen, 0,8 grand).

In [ ]:
# ---- 1.1 Le comportement de chaque participant dans chaque condition (le code : ob.comportement)
comport = ob.comportement(essais, HYPOTHESE, POSTURE)
comport.round(1)                                             # une ligne par participant et condition

In [ ]:
# ---- F1 : le comportement
mesures_c = [m for m in ('TR (ms)', 'Exactitude (%)', 'IES (ms)') if m in comport]   # avec 'tr', le TR seulement
larges = {mesure: comport.pivot(index='participant', columns='condition', values=mesure)[[COND_A, COND_B]].dropna()
          for mesure in mesures_c}
ob.figure_conditions(larges, COULEURS)                       # une barre par condition, un trait par participant
for mesure, large in larges.items():
    t, p = stats.ttest_rel(large[COND_B], large[COND_A])
    difference = large[COND_B] - large[COND_A]
    print(f'{mesure} : {COND_A} {large[COND_A].mean():.1f}, {COND_B} {large[COND_B].mean():.1f} ; '
          f't({len(large) - 1}) = {t:.2f}, p = {p:.3f}, dz = {difference.mean() / difference.std(ddof=1):.2f}')
if HYPOTHESE == 'tr':
    print("(par construction, les essais lents ont un TR plus long : ce test décrit seulement l'écart entre les deux moitiés)")

## 2. Les ERP du groupe

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Le comportement dit si A et B diffèrent ; les ERP disent **quand** le cerveau les traite différemment : les ERP (2.1), l'onde de différence
(2.2), les essais un par un (2.3), et pour aller plus loin, la ligne de base (2.4).

### 2.1 Les ERP du groupe

L'ERP d'un participant = la moyenne de ses époques dans une condition ; l'ERP du groupe = la moyenne des participants (la zone ombrée :
l'intervalle de confiance à 95 % de cette moyenne). Un participant qui a moins de 20 essais dans une condition est laissé de côté. Fond
gris : la fenêtre.

In [ ]:
# ---- 2.1 L'ERP de chaque participant dans chaque condition
erp_A, erp_B, onde_A, onde_B, participants = [], [], [], [], []   # par participant : l'ERP de chaque électrode, et celui de la région
for participant, ep in EPOQUES.items():
    e_a, e_b = ep[CHOIX_A], ep[CHOIX_B]
    if min(len(e_a), len(e_b)) < MIN_EPOQUES:
        print(f'{participant} laissé de côté : {len(e_a)} essais {COND_A}, {len(e_b)} essais {COND_B}')
        continue
    erp_A.append(ob.erp_electrodes(e_a))                     # chaque électrode : pour les cartes (partie 3)
    erp_B.append(ob.erp_electrodes(e_b))
    onde_A.append(REGIONALES[participant][CHOIX_A].average())   # la région : la moyenne de ses essais de A
    onde_B.append(REGIONALES[participant][CHOIX_B].average())
    participants.append(participant)
print(f'{len(participants)} participants : {", ".join(participants)}')

In [ ]:
# ---- F2 : les ERP du groupe
ob.figure_ondes({COND_A: onde_A, COND_B: onde_B}, COULEURS, f'ERP du groupe — {NOM}, région {", ".join(REGION)}', ZERO,
                fenetre=FENETRE)

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> À vous de compléter : la taille de l'effet du groupe

À partir des ERP de la région (`onde_A` et `onde_B` : pour chaque condition, une liste d'`Evoked`, l'objet ERP de MNE, un par
participant), calculez pour chaque participant l'amplitude moyenne de B − A dans `FENETRE`, puis la moyenne du groupe (en µV).

In [ ]:
# ---- Exercice
# effets = [b.copy().crop(*FENETRE).data.mean() * 1e6 - a.copy().crop(...).data.mean() * 1e6     # <--- à compléter
#           for a, b in zip(onde_A, onde_B)]
# print(f"{COND_B} - {COND_A} : {np.mean(effets):+.2f} µV en moyenne, de {min(effets):+.2f} à {max(effets):+.2f}")

In [ ]:
#@title Solution (à consulter après avoir essayé)
effets = [b.copy().crop(*FENETRE).data.mean() * 1e6 - a.copy().crop(*FENETRE).data.mean() * 1e6 for a, b in zip(onde_A, onde_B)]
print(f"{COND_B} - {COND_A} : {np.mean(effets):+.2f} µV en moyenne, de {min(effets):+.2f} à {max(effets):+.2f}")
# Les participants varient beaucoup : c'est pourquoi on teste l'effet sur le groupe, pas sur un participant.

### 2.2 L'onde de différence et le test par amas

L'**onde de différence** = l'ERP de B moins celui de A, instant par instant. Tester chaque instant séparément multiplie les tests, et
donc les faux positifs. Le **test par amas** (Maris et Oostenveld, 2007) regroupe en un **amas** les instants voisins qui dépassent le
seuil, puis compare chaque amas à ceux qu'on obtient par hasard, en inversant au hasard le signe de l'effet des participants :

![amas](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/amas.png)

Dans la figure : en gris clair, l'onde de chaque participant ; en noir, leur moyenne et son intervalle de confiance ; en bleu clair, la
fenêtre de votre hypothèse. Chaque amas porte son p : **vert** s'il est significatif (il dit **qu'il y a** une différence dans cet
intervalle, pas exactement quand elle commence ; Sassenhagen et Draschkow, 2019), **gris** sinon (à ne pas interpréter).

In [ ]:
# ---- F3 : l'onde de différence et le test par amas
from mne.stats import permutation_cluster_1samp_test

ondes_difference = [mne.combine_evoked([b, a], weights=[1, -1]) for a, b in zip(onde_A, onde_B)]   # B - A, chaque participant
X = np.array([d.data[0] for d in ondes_difference]) * 1e6     # participants x instants (µV)
seuil = stats.t.ppf(0.975, len(X) - 1)                        # forme les amas : le t d'un test bilatéral à p < .05
_, amas, p_amas, _ = permutation_cluster_1samp_test(X, threshold=seuil, n_permutations=1024, tail=0, out_type='mask', seed=97)
ob.figure_difference(ondes_difference, FENETRE, amas, p_amas, ZERO, f'{COND_B} − {COND_A}, {NOM}, région {", ".join(REGION)}')
temps = ondes_difference[0].times
ordre = np.argsort([temps[m][0] for m in amas])                # les amas dans l'ordre du temps
print('Amas :', ', '.join(f'{temps[amas[k]][0] * 1000:.0f} à {temps[amas[k]][-1] * 1000:.0f} ms (p = {p_amas[k]:.3f})' for k in ordre) or 'aucun')

**Lire.** Les ondes des participants (en gris) sont-elles du même côté de 0 dans la fenêtre ? Plus elles se dispersent, plus
l'intervalle de confiance de la moyenne est large. Un amas vert dans la fenêtre soutient l'hypothèse ; aucun amas, ou des amas gris :
pas de différence que le test puisse distinguer du hasard.

### <span style="background:#7C3AED; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 3</span> À vous de construire : l'onde de différence de chaque participant

Objectif : une grille de petites figures, une par participant, avec son onde de différence B − A dans la région, et la fenêtre en gris.

<div style="border-left:5px solid #7C3AED; background:#F5F0FF; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#7C3AED;">Indices</b><br><ul style='margin:4px 0 0 0;'><li>Les ondes de différence sont dans <code>ondes_difference</code> (une liste d'<code>Evoked</code>, dans l'ordre de <code>participants</code>).</li><li><code>fig, axes = plt.subplots(3, 6, figsize=(15, 6), sharex=True, sharey=True)</code>, puis <code>axes.ravel()</code>.</li><li>Pour chaque onde : <code>ax.plot(d.times, d.data[0] * 1e6)</code> et <code>ax.axvspan(*FENETRE, color='0.9')</code>.</li></ul></div>

In [ ]:
# ---- Votre cellule

In [ ]:
#@title Solution (à consulter après avoir essayé)
fig, axes = plt.subplots(3, 6, figsize=(15, 6), sharex=True, sharey=True)
for ax, participant, d in zip(axes.ravel(), participants, ondes_difference):
    ax.axvspan(*FENETRE, color="0.9")
    ax.axhline(0, color="0.6", lw=0.6)
    ax.plot(d.times, d.data[0] * 1e6, color="k", lw=1)
    ax.set_title(participant, fontsize=9)
for ax in axes.ravel()[len(participants):]:
    ax.set_axis_off()
fig.supxlabel(f"Temps depuis {ZERO} (s)")
fig.supylabel(f"{COND_B} - {COND_A} (µV)")
plt.tight_layout()
plt.show()

### 2.3 Les essais un par un

Chaque ligne de l'image est un essai de `PARTICIPANT` (couleur : le signal de la région), du TR le plus court (en bas) au plus long ; le
trait noir marque l'autre événement (la tape, ou le mot) ; en dessous, leur moyenne. Une onde liée au **zéro** forme une colonne
verticale ; une onde liée à **l'autre** événement suit le trait.

In [ ]:
# ---- 2.3 L'image des essais d'un participant
essais_image = mne.concatenate_epochs([REGIONALES[PARTICIPANT][CHOIX_A], REGIONALES[PARTICIPANT][CHOIX_B]])   # les deux conditions
ob.figure_image_essais(essais_image, CENTRE, ZERO, f'{PARTICIPANT} : {COND_A} et {COND_B}, région {", ".join(REGION)}')

### <span style="background:#5B6472; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Pour aller plus loin</span> <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> 2.4 La ligne de base par régression

La **ligne de base** d'un essai est son signal moyen juste avant le mot (−0,2 à 0 s). La **soustraction** (le défaut) la retire telle
quelle : elle suppose qu'elle pèse de la même façon à chaque instant (une pente de 1). La **régression** (Alday, 2019) estime cette pente
β(t) à chaque instant et à chaque électrode, et retire β(t) × la ligne de base :

![ligne de base](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/ligne_de_base.png)

**À faire.** Dans 0.4, mettez `LIGNE_DE_BASE = 'regression'`, relancez à partir de 0.5, puis exécutez la cellule ci-dessous. Une pente
proche de 1 : la soustraction suffisait. Comparez ensuite l'effet de la partie 4 avec les deux corrections : votre conclusion change-t-elle ?

In [ ]:
# ---- 2.4 La pente de la ligne de base : dans la région au cours du temps, et sur la tête dans la fenêtre
if LIGNE_DE_BASE == 'soustraction':
    print("Avec la soustraction, la pente vaut 1 partout : mettez LIGNE_DE_BASE = 'regression' dans 0.4, puis relancez à partir de 0.5.")
else:
    pente_region = ob.pentes_region(PENTES, REGION)          # la pente de chaque participant, moyennée dans la région
    ob.figure_pente(PENTES, pente_region, REGION, FENETRE, ZERO)
    dans_fenetre = [p.copy().crop(*FENETRE).get_data().mean() * 1e6 for p in pente_region]
    print(f'Pente dans la région, de {FENETRE[0] * 1000:.0f} à {FENETRE[1] * 1000:.0f} ms : médiane {np.median(dans_fenetre):.2f}, '
          f'de {min(dans_fenetre):.2f} à {max(dans_fenetre):.2f} selon le participant (la soustraction suppose 1)')

## 3. Les topographies

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Après le décours dans le temps (partie 2), la répartition sur la tête. Une **topographie** est la carte de l'amplitude moyenne dans la
fenêtre, vue de dessus (le nez en haut) : A, B, puis B − A. Points noirs : la région. Cercles vides : électrodes mesurées chez trop peu
de participants (moins de 3, ou de la moitié). Ronds blancs sur la différence : p < .05 à cette électrode (non corrigé pour le nombre de
tests).

In [ ]:
# ---- F4 : les topographies
valeurs_A = np.array([e.copy().crop(*FENETRE).get_data(units='uV').mean(axis=1) for e in erp_A])   # participants x électrodes (µV ; NaN : pas mesurée)
valeurs_B = np.array([e.copy().crop(*FENETRE).get_data(units='uV').mean(axis=1) for e in erp_B])
deux = np.isfinite(valeurs_A) & np.isfinite(valeurs_B)       # le participant a mesuré l'électrode dans les deux conditions
vue = deux.sum(axis=0) >= max(3, len(participants) / 2)      # la règle du projet : au moins 3 participants, et la moitié d'entre eux
visibles = np.flatnonzero(vue)
p_electrodes = np.array([stats.ttest_rel(valeurs_B[deux[:, j], j], valeurs_A[deux[:, j], j]).pvalue for j in visibles])   # un test par électrode
groupe, info_vue = ob.erp_groupe_visibles(erp_A, erp_B, deux, vue, COND_A, COND_B)   # l'ERP du groupe à chaque électrode visible : A, B et B - A
ob.figure_topographies(groupe, erp_A[0].info, vue, REGION, p_electrodes, FENETRE,
                       f'Topographies — {NOM}, {FENETRE[0] * 1000:.0f} à {FENETRE[1] * 1000:.0f} ms depuis {ZERO}')
noms_ch = np.array(erp_A[0].ch_names)
print('Électrodes à p < .05 (non corrigé) :', ', '.join(noms_ch[vue][p_electrodes < 0.05]) or 'aucune')
print('Électrodes trop peu mesurées pour les cartes (cercles vides) :', ', '.join(noms_ch[~vue]) or 'aucune')

### 3.1 Le test par amas sur toute la tête

Le même principe que 2.2, sur toutes les électrodes et tous les instants à la fois (0 à 0,8 s après le mot ; toute l'époque autour de la
tape) : un amas regroupe des points voisins dans le temps **et** sur la tête. Il ne fixe ni la région ni la fenêtre à l'avance. La partie
5 peut mesurer l'effet dans l'amas le plus fort (`ZONE = 'amas'`).

À gauche : le t de B − A pour chaque électrode (une ligne, l'avant de la tête en haut) à chaque instant ; en gris, hors des amas
significatifs. À droite : la carte de l'amas le plus fort, ses électrodes en blanc.

In [ ]:
# ---- F5 : le test par amas sur toutes les électrodes et tous les instants
temps = erp_A[0].times
testes = (temps >= -1e-9) & (temps <= 0.8 + 1e-9) if CENTRE == 'mot' else np.ones(len(temps), bool)   # les instants testés
temps_testes = temps[testes]
# participants x instants x électrodes visibles (µV) : la différence B - A de chaque participant, NaN là où l'électrode n'est pas mesurée
X = np.array([(b.data - a.data)[visibles][:, testes].T for a, b in zip(erp_A, erp_B)]) * 1e6
adjacence, _ = mne.channels.find_ch_adjacency(info_vue, ch_type='eeg')   # les électrodes voisines sur la tête
seuil = stats.t.ppf(0.975, len(X) - 1)                                   # forme les amas : le t d'un test bilatéral à p < .05
t_obs, amas, p_amas, _ = mne.stats.spatio_temporal_cluster_1samp_test(   # ob.t_mesures : le t avec les seuls participants mesurés
    X, threshold=seuil, n_permutations=5000, tail=0, stat_fun=ob.t_mesures, adjacency=adjacence, out_type='mask', seed=97)
AMAS_ELECTRODES, AMAS_FENETRE = REGION, FENETRE                         # sans amas, la partie 5 garde REGION et FENETRE
if len(amas):
    AMAS_ELECTRODES, AMAS_FENETRE = ob.amas_le_plus_fort(amas, p_amas, temps_testes, info_vue.ch_names)   # ses électrodes, ses instants
ob.figure_amas_tete(t_obs, amas, p_amas, temps_testes, info_vue, groupe[f'{COND_B} - {COND_A}'], f'{COND_B} - {COND_A}', ZERO)
for k in np.argsort(p_amas)[:3]:                                         # les trois amas les plus forts
    instants_k = temps_testes[amas[k].any(axis=1)]
    print(f'Amas {"positif" if t_obs[amas[k]].sum() > 0 else "négatif"} de {instants_k[0] * 1000:.0f} à {instants_k[-1] * 1000:.0f} ms, '
          f'{", ".join(np.array(info_vue.ch_names)[amas[k].any(axis=0)])} : p = {p_amas[k]:.3f}')

**Lire.** L'amas le plus fort recouvre-t-il la région et la fenêtre de votre hypothèse ? S'il est ailleurs, est-ce plausible pour le
cerveau, ou plutôt un artefact (les yeux à l'avant, la réponse de la main au-dessus de C3 et C4 juste avant la tape) ?

## 4. Amplitude, latence et SME : le test de votre hypothèse

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

Pour tester l'effet dans la région et la fenêtre choisies, on résume l'ERP de chaque participant par des nombres :

![mesures](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/mesures.png)

- **amplitude moyenne** : la moyenne de l'ERP dans la fenêtre (la mesure principale ; **le test de votre rapport**) ;
- **latence du pic** : l'instant du point le plus haut (ou le plus bas) dans une fenêtre de recherche plus large (le fond gris de la
  figure de droite) ;
- **SME** (erreur de mesure standardisée) : la précision de l'amplitude d'un participant ; plus elle est petite, plus la mesure est fiable.

Pour la N400, une amplitude **plus négative** = une onde plus grande.

In [ ]:
# ---- 4.1 Amplitude, latence et SME de chaque participant
lignes = []
for participant, a, b in zip(participants, onde_A, onde_B):
    for condition, onde, choix in ((COND_A, a, CHOIX_A), (COND_B, b, CHOIX_B)):
        amplitude = onde.copy().crop(*FENETRE).get_data(units='uV').mean()
        # strict=False : le point le plus haut (ou le plus bas) de la fenêtre de recherche, quel que soit son signe
        _, latence = onde.get_peak(tmin=FENETRE_LATENCE[0], tmax=FENETRE_LATENCE[1], mode=POLARITE, strict=False)
        par_essai = REGIONALES[participant][choix].crop(*FENETRE).get_data(units='uV').mean(axis=(1, 2))   # chaque essai
        lignes.append({'participant': participant, 'condition': condition, 'amplitude (µV)': amplitude, 'latence (ms)': latence * 1000,
                       'SME (µV)': par_essai.std(ddof=1) / np.sqrt(len(par_essai)), 'essais': len(par_essai)})
mesures_eeg = pd.DataFrame(lignes)
mesures_eeg.round(2)

In [ ]:
# ---- F6 : amplitude et latence ; le test t apparié de B contre A
larges = {mesure: mesures_eeg.pivot(index='participant', columns='condition', values=mesure)[[COND_A, COND_B]]
          for mesure in ['amplitude (µV)', 'latence (ms)']}
ob.figure_conditions(larges, COULEURS, prefixe=f'{NOM} : ', largeur=4.5, recherche_latence=FENETRE_LATENCE)   # gris : la recherche du pic
for mesure, large in larges.items():
    t, p = stats.ttest_rel(large[COND_B], large[COND_A])
    difference = large[COND_B] - large[COND_A]
    print(f'{NOM} {mesure} : {COND_A} {large[COND_A].mean():.2f}, {COND_B} {large[COND_B].mean():.2f} ; '
          f't({len(large) - 1}) = {t:.2f}, p = {p:.3f}, dz = {difference.mean() / difference.std(ddof=1):.2f}')

### <span style="background:#5B6472; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Pour aller plus loin</span> <span style="background:#7C3AED; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 3</span> 4.2 Une latence plus robuste : la latence à 50 % de l'aire

La latence du pic dépend d'un seul point, souvent du bruit. La **latence à 50 % de l'aire** (Luck, 2014) prend l'instant où l'aire sous
la courbe, dans la fenêtre de recherche, atteint la moitié de son total : elle utilise toute l'onde. Objectif : la calculer pour chaque
participant et chaque condition, puis la comparer entre A et B.

<div style="border-left:5px solid #7C3AED; background:#F5F0FF; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#7C3AED;">Indices</b><br><ul style='margin:4px 0 0 0;'><li>Pour une onde : <code>x = onde.copy().crop(*FENETRE_LATENCE)</code>, puis <code>x.get_data(units='uV')[0]</code> et <code>x.times</code>.</li><li>Gardez seulement le côté de la composante : <code>np.clip(v, 0, None)</code> pour un pic positif, <code>np.clip(-v, 0, None)</code> pour la N400 (<code>POLARITE</code>).</li><li>L'aire cumulée : <code>np.cumsum(...)</code> ; l'instant où elle dépasse la moitié : <code>np.searchsorted(cumul, cumul[-1] / 2)</code>.</li><li>Puis <code>stats.ttest_rel</code>, comme dans F6.</li></ul></div>

In [ ]:
# ---- Votre cellule

In [ ]:
#@title Solution (à consulter après avoir essayé)
def latence_aire(onde, fenetre=FENETRE_LATENCE, polarite=POLARITE, part=0.5):
    """L'instant (ms) où l'aire sous la courbe, du côté de la composante, atteint part de son total dans la fenêtre."""
    x = onde.copy().crop(*fenetre)
    v = x.get_data(units='uV')[0]
    cumul = np.cumsum(np.clip(v if polarite == 'pos' else -v, 0, None))
    return x.times[np.searchsorted(cumul, part * cumul[-1])] * 1000


aire = pd.DataFrame({COND_A: [latence_aire(a) for a in onde_A], COND_B: [latence_aire(b) for b in onde_B]}, index=participants)
t, p = stats.ttest_rel(aire[COND_B], aire[COND_A])
print(f"Latence à 50 % de l'aire : {COND_A} {aire[COND_A].mean():.0f} ms, {COND_B} {aire[COND_B].mean():.0f} ms ; "
      f"t({len(aire) - 1}) = {t:.2f}, p = {p:.3f}")
print(f"Écart-type entre participants : pic {larges['latence (ms)'].std().mean():.0f} ms, 50 % de l'aire {aire.std().mean():.0f} ms")
# Une mesure plus robuste varie moins d'un participant à l'autre pour des raisons de bruit.

## 5. L'interaction avec la posture

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H4</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H8</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H10</span></div>

![interaction](https://raw.githubusercontent.com/thecocolab/PSY2007D-UdeM/main/figures/brainwalk_2_erp/interaction.png)

L'**interaction** : l'effet change-t-il avec la posture ? L'effet de fréquence (LF − HF) est mesuré assis et en marche pour chaque
participant, puis le test t apparié compare les deux postures. Pour H10, c'est le second volet : l'effet de fréquence **sur la CPP**.

Les trois réglages de la cellule :

- `EFFET` : l'effet comparé entre les postures, `'frequence'` (LF − HF, H4, H8, H10) ou `'lexicalite'` (pseudo-mots − mots, pour explorer) ;
- `ZONE` : où le mesurer, `'region'` (votre région et votre fenêtre) ou `'amas'` (l'amas le plus fort de 3.1 ; attention, voir plus bas) ;
- `COMPORTEMENT` : le même effet sur le comportement, `'TR (ms)'`, `'Exactitude (%)'` ou `'IES (ms)'`.

Dans la figure, en haut : l'effet dans chaque posture, l'écart marche − assis, puis l'effet de chaque participant (EEG, puis
comportement) ; en bas : les cartes de l'effet assis, en marche, et de l'écart.

In [ ]:
# ---- 5.1 L'effet dans chaque posture (le code : ob.effet_par_posture)
EFFET = 'frequence'            # 'frequence' (LF - HF) ou 'lexicalite' (pseudo-mots - mots)
ZONE = 'region'                # 'region' (REGION et FENETRE) ou 'amas' (l'amas le plus fort de 3.1)
COMPORTEMENT = 'TR (ms)'       # 'TR (ms)', 'Exactitude (%)' ou 'IES (ms)'

NOM_EFFET = 'LF - HF' if EFFET == 'frequence' else 'pseudo-mots - mots'
ELECTRODES_5, FENETRE_5 = (AMAS_ELECTRODES, AMAS_FENETRE) if ZONE == 'amas' else (REGION, FENETRE)
interaction, ondes, cartes = ob.effet_par_posture(EPOQUES, EFFET, ELECTRODES_5, FENETRE_5)   # EEG : un participant par ligne
interaction_c = ob.effet_comportement_par_posture(essais, EFFET, COMPORTEMENT)             # le comportement
print(f'{NOM} : effet {NOM_EFFET} (µV), {", ".join(ELECTRODES_5)}, de {FENETRE_5[0] * 1000:.0f} à {FENETRE_5[1] * 1000:.0f} ms')
interaction.round(2)

In [ ]:
# ---- F7 : l'effet dans chaque posture, et l'interaction (marche - assis)
ob.figure_interaction(ondes, cartes, interaction, interaction_c, ELECTRODES_5, FENETRE_5, NOM_EFFET, COMPORTEMENT, ZERO)
for nom, tableau, unite in ((f'EEG, {NOM}', interaction, 'µV'), (COMPORTEMENT, interaction_c, COMPORTEMENT.split('(')[1].rstrip(')'))):
    n = len(tableau)
    for posture in ('assis', 'marche'):                      # l'effet dans chaque posture, contre 0
        t, p = stats.ttest_1samp(tableau[posture], 0.0)
        print(f'{nom} : effet {posture} {tableau[posture].mean():+.2f} {unite}, t({n - 1}) = {t:.2f}, p = {p:.3f}')
    ecart = tableau['marche'] - tableau['assis']             # l'interaction : l'effet change-t-il avec la posture ?
    t, p = stats.ttest_rel(tableau['marche'], tableau['assis'])
    print(f'{nom} : interaction (marche - assis) {ecart.mean():+.2f} {unite}, t({n - 1}) = {t:.2f}, p = {p:.3f}, '
          f'dz = {ecart.mean() / ecart.std(ddof=1):.2f}\n')

**Lire.** L'effet change-t-il avec la posture, dans l'EEG et dans le comportement ? Sa carte en marche ressemble-t-elle à sa carte assise ?
Avec `ZONE = 'amas'` et `POSTURE = 'assis'`, l'effet assis est mesuré là où il a été trouvé : il est surestimé, et l'interaction aussi.

## 6. L'EEG et le comportement

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H5</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H9</span></div>

Un point par participant : par défaut son effet sur le TR (horizontal) et son effet sur l'amplitude (vertical), dans la région et la
fenêtre de la partie 4. Le **r de Pearson** mesure la force du lien linéaire (de −1 à 1 ; 0 : aucun lien). Avec 17 participants, c'est
exploratoire. Pour H9, l'écart de TR entre lents et rapides existe par construction : la question est de savoir si les participants dont
l'écart est grand ont aussi une plus grande différence de CPP.

Trois réglages : `COMPORTEMENT` (la mesure du comportement ; avec H9, le TR seulement), `MESURE_EEG` (`'amplitude (µV)'` ou
`'latence (ms)'`, partie 4), et `VALEURS` (`'difference'` : l'effet B − A de chaque participant ; ou une condition, par exemple `COND_A`).

In [ ]:
# ---- 6.1 Le comportement et l'EEG de chaque participant
COMPORTEMENT = 'TR (ms)'       # 'TR (ms)', 'Exactitude (%)' ou 'IES (ms)'
MESURE_EEG = 'amplitude (µV)'  # 'amplitude (µV)' ou 'latence (ms)'
VALEURS = 'difference'         # 'difference' ou une condition, par exemple COND_A

large_c = comport.pivot(index='participant', columns='condition', values=COMPORTEMENT)
large_e = mesures_eeg.pivot(index='participant', columns='condition', values=MESURE_EEG)
if VALEURS == 'difference':
    x, y = large_c[COND_B] - large_c[COND_A], large_e[COND_B] - large_e[COND_A]   # l'effet de chaque participant
else:
    x, y = large_c[VALEURS], large_e[VALEURS]                                     # les valeurs d'une condition
points = pd.DataFrame({'comportement': x, 'eeg': y}).dropna()                     # les participants qui ont les deux
points.round(2)

In [ ]:
# ---- F8 : le nuage de points
r, p = stats.pearsonr(points['comportement'], points['eeg'])
suffixe = f' ({COND_B} - {COND_A})' if VALEURS == 'difference' else f' ({VALEURS})'
ob.figure_nuage(points, COMPORTEMENT + suffixe, f'{NOM} : {MESURE_EEG}' + suffixe)   # un point par participant, et la droite
print(f'Pearson : r = {r:.2f}, p = {p:.3f} (n = {len(points)})')

### <span style="background:#E8710A; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Type 2</span> À vous de compléter : un lien robuste ?

Un seul participant extrême peut faire (ou défaire) un r de Pearson. Le **rho de Spearman** compare les **rangs** des participants : il
résiste mieux aux valeurs extrêmes. Calculez-le avec `stats.spearmanr`, puis refaites Pearson sans le participant le plus éloigné.

In [ ]:
# ---- Exercice
# rho, p_rho = stats.spearmanr(points['comportement'], ...)                     # <--- à compléter
# extreme = ((points - points.mean()) / points.std()).abs().max(axis=1).idxmax()  # le participant le plus éloigné
# r_sans, p_sans = stats.pearsonr(*points.drop(index=...).T.to_numpy())         # <--- à compléter

In [ ]:
#@title Solution (à consulter après avoir essayé)
rho, p_rho = stats.spearmanr(points['comportement'], points['eeg'])
extreme = ((points - points.mean()) / points.std()).abs().max(axis=1).idxmax()
r_sans, p_sans = stats.pearsonr(*points.drop(index=extreme).T.to_numpy())
print(f'Spearman : rho = {rho:.2f}, p = {p_rho:.3f} ; Pearson sans {extreme} : r = {r_sans:.2f}, p = {p_sans:.3f}')
# Si le lien disparaît sans un seul participant, il ne faut pas le présenter comme un résultat.

## 7. À retenir pour votre rapport

| Votre question | Hypothèses | Le test principal | Les figures |
|---|---|---|---|
| A et B diffèrent-ils ? | H1, H2, H3, H6, H7, H9, H10 | le test t apparié sur l'amplitude moyenne (partie 4) | les ERP et l'onde de différence (2), la carte (3) |
| L'effet change-t-il en marche ? | H4, H8, H10 | le test t apparié sur l'effet, marche contre assis (partie 5) | l'effet dans chaque posture (5) |
| L'EEG suit-il le comportement ? | H5, H9 | le r de Pearson (partie 6) | le nuage de points (6) |

<div style="border-left:5px solid #15803D; background:#ECF8EF; color:#1F2937; padding:10px 16px; border-radius:6px; margin:8px 0;"><b style="color:#15803D;">À retenir</b><br><ul style='margin:4px 0 0 0;'><li>Énoncez l'hypothèse et vos prédictions (mesure, condition, sens de l'effet) <b>avant</b> les résultats.</li><li>Pour chaque test : la moyenne de chaque condition, t(ddl), p, dz et le nombre de participants.</li><li>Une interaction se teste directement (partie 5) ; deux p séparés ne la testent pas.</li><li>Un résultat nul est un résultat : rapportez-le avec la même précision.</li></ul></div>

**Pistes pour aller plus loin** : votre effet tient-il quand on change un choix raisonnable ? Refaites les parties 0.5 à 4 en changeant
**une** option à la fois (0.4), et notez t, p et dz dans un tableau :

| Changement | Pourquoi |
|---|---|
| `POSTURE = 'assis'`, puis `'marche'` | l'effet existe-t-il dans chaque posture ? |
| `SANS_CLIGNEMENT = True` | pour la P3/LPC et la CPP : un clignement suit souvent la réponse |
| `EXIGER_MINIMUM = True` | les époques mal mesurées pèsent-elles sur l'effet ? |
| `LIGNE_DE_BASE = 'regression'` | la correction de la ligne de base (2.4) |
| la latence à 50 % de l'aire (4.2) | une mesure de latence moins sensible au bruit |

Un effet qui tient dans tous les cas est plus convaincant ; un effet qui n'apparaît que dans un cas ne doit pas être présenté seul.

**Limites à discuter** : 17 participants ; les tests non corrigés (instant par instant, électrode par électrode) ; la fragilité de la
latence du pic ; les clignements autour de la tape ; les électrodes sèches et la marche ; le passe-haut à 1 Hz (notebook 1), choisi pour
la N400, qui déforme les ondes lentes comme la P3/LPC ; les deux postures nettoyées par des chaînes un peu différentes (notebook 1,
partie 11). Pour H9 : autour de la tape, l'ERP du mot ne tombe pas au même moment dans les essais rapides et lents (le mot est environ
200 ms plus loin pour un essai lent) ; une partie de la différence de CPP peut venir de là.

## 8. Défis (points bonus)

Pour les groupes qui veulent aller plus loin : cinq défis, avec les **outils de ce notebook** (une option de 0.4, une cellule à relancer,
un tableau à relire). Il n'y a pas de solution : c'est votre travail. Pour chacun, présentez dans les **annexes du rapport** :

1. la question, et pourquoi elle compte pour votre hypothèse ;
2. ce que vous avez changé (les options, les fenêtres, les participants) ;
3. le résultat (t, p, dz, ou une figure), et votre interprétation en deux ou trois phrases.

**Une règle** : remettez les options à leur valeur de départ (0.4) après chaque défi, et relancez à partir de 0.5.

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> A. La P3/LPC, alignée sur le mot ou sur la réponse ?

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H6</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H7</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H8</span></div>

**La question.** Les mots rares sont reconnus plus lentement (environ 75 ms). Si la P3/LPC suit la **réponse** plutôt que le mot, elle
arrive plus tard pour LF, et la moyenne alignée sur le mot l'étale : une différence LF − HF peut alors venir seulement du moment de la
réponse (partie 2.3 : l'image des essais). On mesure donc le même effet **juste avant la tape**.

**Comment.**

1. Dans 0.4, ajoutez la ligne `COMPOSANTE, FENETRE = 'P3/LPC (tape)', (-0.2, 0.0)` : les époques de la tape, les 200 ms avant la réponse.
2. Relancez à partir de 0.5, jusqu'à la partie 4 (F2 et F6).

**À rapporter.** L'effet (t, p, dz) aligné sur le mot (votre résultat) et aligné sur la réponse. Lequel est le plus net ? Que dit-il de
votre hypothèse : un effet de la **reconnaissance du mot**, ou du **moment de la réponse** ?

In [ ]:
# ---- Défi A : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> B. Un contrôle positif : le notebook trouve-t-il un effet connu ?

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

**La question.** Un résultat nul peut venir d'un effet absent, ou d'une analyse qui ne voit rien (trop de bruit, trop peu de
participants). Pour le savoir, on lance la même analyse sur un effet **connu** : les pseudo-mots donnent une N400 plus négative que les
mots (Kutas et Federmeier, 2011).

**Comment.**

1. Dans 0.4, ajoutez la ligne `HYPOTHESE, COMPOSANTE, FENETRE, REGION = 'lexicalite', 'N400', (0.30, 0.50), CP`.
2. Relancez à partir de 0.5 jusqu'à la partie 4.

**À rapporter.** L'effet de lexicalité (t, p, dz) à côté du vôtre. Le notebook le trouve-t-il ? S'il le trouve, un résultat nul pour votre
hypothèse devient plus crédible ; s'il ne le trouve pas non plus, qu'en conclure sur la sensibilité de l'analyse (17 participants,
électrodes sèches, la marche) ?

In [ ]:
# ---- Défi B : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> C. Une fenêtre choisie sans tricher

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H2</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H3</span> <span style="background:#0B2545; color:#FFFFFF; border-radius:10px; padding:1px 8px; font-size:0.85em;">H7</span></div>

**La question.** Choisir la fenêtre là où la différence est la plus grande, puis la tester sur les **mêmes** données, rend le test trop
optimiste (le « double dipping »). La solution : choisir la fenêtre sur une partie des données, et la tester sur une **autre**. Ici, on
la choisit sur les essais **assis** et on la teste sur les essais **en marche**.

**Comment.**

1. Dans 0.4, `POSTURE = 'assis'` ; relancez jusqu'à F3. Trouvez l'instant où l'onde de différence moyenne est la plus loin de 0,
   dans la fenêtre de recherche de votre composante :
   `moyenne = np.mean([d.data[0] for d in ondes_difference], axis=0)` (l'onde de différence du groupe, cellule F3),
   `t = ondes_difference[0].times`, `dedans = (t >= FENETRE_LATENCE[0]) & (t <= FENETRE_LATENCE[1])`, puis
   `pic = t[dedans][np.argmax(np.abs(moyenne)[dedans])]`.
2. Dans 0.4, `POSTURE = 'marche'` et `FENETRE = (pic - 0.05, pic + 0.05)` ; relancez jusqu'à la partie 4.

**À rapporter.** La fenêtre choisie (assis), et l'effet dans cette fenêtre en marche (t, p, dz). L'effet se retrouve-t-il dans des
données qui n'ont pas servi à le choisir ?

In [ ]:
# ---- Défi C : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> D. À gauche ou à droite ?

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

**La question.** Les régions du notebook sont **symétriques** : elles mêlent les deux hémisphères. Or un effet peut être plus net d'un
côté : pour des mots écrits, la N400 est souvent un peu plus grande au-dessus de l'hémisphère **droit** (Kutas et Federmeier, 2011),
même si le langage est surtout traité à gauche (une carte de surface ne dit pas directement où se trouve la source).

**Comment.**

1. Dans 0.4, `REGION = ['C3', 'P3']` (l'hémisphère gauche ; pour la P200 : `['F3', 'C3', 'P3']`) ; relancez jusqu'à la partie 4, et
   gardez l'effet de chaque participant : `gauche = larges['amplitude (µV)'][COND_B] - larges['amplitude (µV)'][COND_A]` (cellule F6).
2. Même chose avec `REGION = ['C4', 'P4']` (ou `['F4', 'C4', 'P4']`) : `droite = ...`.
3. Comparez : `stats.ttest_rel(gauche, droite)`.

**À rapporter.** L'effet à gauche, à droite, et leur différence (t, p). Avec deux électrodes par côté, chaque mesure est plus bruitée :
tenez-en compte. Attention, pour H9 et H10 : avant la tape, la main qui répond active C3 ou C4 (la symétrie des régions sert justement
à annuler cela).

In [ ]:
# ---- Défi D : votre cellule (gardez-y les résultats que vous rapportez)

### <span style="background:#B45309; color:#FFFFFF; border-radius:12px; padding:2px 10px; font-size:0.72em; vertical-align:middle;">Défi</span> E. Seulement les participants bien mesurés

<div style="margin:2px 0 8px 0; color:#5B6472;">Pour : <span style="color:#0B5CAD; font-weight:bold;">toutes les hypothèses</span></div>

**La question.** La SME (partie 4) dit la précision de la mesure de chaque participant. Un participant très bruité (électrodes sèches,
peu d'essais) pèse autant que les autres dans le test t. L'effet tient-il sans les participants les moins bien mesurés ?

**Comment.**

1. À partir de `mesures_eeg` (cellule 4.1) : la SME moyenne de chaque participant,
   `sme = mesures_eeg.groupby('participant')['SME (µV)'].mean()`.
2. Écartez les participants au-dessus d'un seuil **décidé à l'avance** (par exemple 1,5 fois la médiane : `sme > 1.5 * sme.median()`).
3. Refaites le test t de F6 avec les participants restants (`larges['amplitude (µV)'].drop(index=...)`).

**À rapporter.** Les participants écartés et pourquoi (leur SME, leur nombre d'essais), puis l'effet avec et sans eux (t, p, dz). Un
effet qui ne tient qu'avec, ou qu'après les avoir écartés, doit être présenté avec prudence.

In [ ]:
# ---- Défi E : votre cellule (gardez-y les résultats que vous rapportez)

**Suite** : `brainwalk_3_rythmes.ipynb` (les rythmes et le temps-fréquence).

<details><summary>Références</summary>

- Alday, P. M. (2019). How much baseline correction do we need in ERP research? Extended GLM model can replace baseline correction while lifting its limits. *Psychophysiology, 56*(12), e13451.
- Kutas, M., & Federmeier, K. D. (2011). Thirty years and counting: Finding meaning in the N400 component of the event-related brain potential (ERP). *Annual Review of Psychology, 62*, 621-647.
- Luck, S. J. (2014). *An introduction to the event-related potential technique* (2e éd.). MIT Press.
- Luck, S. J., Stewart, A. X., Simmons, A. M., & Rhemtulla, M. (2021). Standardized measurement error: A universal metric of data quality for averaged event-related potentials. *Psychophysiology, 58*, e13793.
- Maris, E., & Oostenveld, R. (2007). Nonparametric statistical testing of EEG- and MEG-data. *Journal of Neuroscience Methods, 164*, 177-190.
- O'Connell, R. G., Dockree, P. M., & Kelly, S. P. (2012). A supramodal accumulation-to-bound signal that determines perceptual decisions in humans. *Nature Neuroscience, 15*, 1729-1735.
- Sassenhagen, J., & Draschkow, D. (2019). Cluster-based permutation tests of MEG/EEG data do not establish significance of effect latency or location. *Psychophysiology, 56*(6), e13335.
- Twomey, D. M., Murphy, P. R., Kelly, S. P., & O'Connell, R. G. (2015). The classic P300 encodes a build-to-threshold decision variable. *European Journal of Neuroscience, 42*, 1636-1643.
</details>